#Semantic Analysis Tools
Taking in stream of words in the form of pkl file. Applying cosine similarity, then placing it on a table.

##Setup

In [3]:
!mkdir embeddings
!cd embeddings
#!curl -o eng-fiction-all_sgns.zip http://snap.stanford.edu/historical_embeddings/eng-fiction-all_sgns.zip
#!unzip eng-fiction-all_sgns.zip
#!mv sgns eng-fiction-all_sgns

!curl -o eng-fiction-all.zip http://snap.stanford.edu/historical_embeddings/eng-fiction-all.zip
!unzip eng-fiction-all.zip


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 2503M  100 2503M    0     0  61.3M      0  0:00:40  0:00:40 --:--:-- 89.3M
Archive:  eng-fiction-all.zip
   creating: eng-fiction-all/
   creating: eng-fiction-all/netstats/
  inflating: eng-fiction-all/netstats/full-nstop_nproper-top10000.pkl  
   creating: eng-fiction-all/svd/
  inflating: eng-fiction-all/svd/1860-vocab.pkl  
  inflating: eng-fiction-all/svd/1900-vocab.pkl  
  inflating: eng-fiction-all/svd/1860-w.npy  
  inflating: eng-fiction-all/svd/1900-w.npy  
  inflating: eng-fiction-all/svd/1840-w.npy  
  inflating: eng-fiction-all/svd/1920-w.npy  
  inflating: eng-fiction-all/svd/1890-w.npy  
  inflating: eng-fiction-all/svd/1970-vocab.pkl  
  inflating: eng-fiction-all/svd/1810-vocab.pkl  
  inflating: eng-fiction-all/svd/1940-w.npy  
  inflating: eng-fiction-all/svd/1920-vocab.pkl  
  inflating: eng-fiction-all/svd

In [4]:
!pip install --upgrade gensim
!pip install openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 57.9 MB/s eta 0:00:00


In [5]:
import numpy as np
import pickle as p
from openai import OpenAI
from google.colab import userdata
from gensim.models import KeyedVectors
import pandas as pd
import random
from IPython.display import display, Markdown

Find minimum frequency words.

In [6]:
with open(f"/content/eng-fiction-all/word_lists/full-nstop_nproper.pkl", "rb") as file:
    clean_words_sorted = p.load(file, encoding="latin1")

with open(f"/content/eng-fiction-all/freqs.pkl", "rb") as file:
    freq_of_words = p.load(file, encoding="latin1")

with open(f"/content/eng-fiction-all/volstats/vols.pkl", "rb") as file:
    score_of_words = p.load(file, encoding="latin1")

In [7]:
selected_decades = [decade for decade in range(1950, 1990+10, 10)]
print(selected_decades)

[1950, 1960, 1970, 1980, 1990]


In [8]:
top_freq_words = clean_words_sorted[:10000]
sanitized_top_freq_words = top_freq_words

In [9]:
for word in score_of_words:
  for decade in selected_decades:
    if np.isnan(score_of_words[word][decade]):
      if word in sanitized_top_freq_words:
        sanitized_top_freq_words.remove(word)

In [10]:
def avg_angular_shift_distance(key):
  cumulative_distance_shift = 0
  for decade in selected_decades:
    cumulative_distance_shift += np.arccos(np.clip(score_of_words[key][decade],-1, 1))
  avg_shift_distance = cumulative_distance_shift/len(selected_decades)
  return avg_shift_distance

In [11]:
shift_list = []

for word in sanitized_top_freq_words:
  shift_list.append([word, avg_angular_shift_distance(word)])

shift_list = sorted(shift_list,key=lambda x: x[1], reverse=True)


In [12]:
print(shift_list)
print(len(shift_list))

percentile = 0.25

low_shift_words = shift_list[int(len(shift_list)-len(shift_list)*percentile):]
high_shift_words = shift_list[:int(len(shift_list)*percentile)]

print(low_shift_words)
print(high_shift_words)
print(len(low_shift_words),len(high_shift_words))

[['wherein', np.float64(1.0592824037683477)], ['invariably', np.float64(1.0380107762865807)], ['likewise', np.float64(1.0317762059178455)], ['doubtless', np.float64(1.0284287033443518)], ['termed', np.float64(1.016898483526426)], ['speedily', np.float64(1.0107858998691188)], ['sombre', np.float64(0.9952735129375126)], ['flora', np.float64(0.9927531314376881)], ['dick', np.float64(0.9920668464134084)], ['travellers', np.float64(0.9920098299343003)], ['captive', np.float64(0.9907007008999514)], ['seemingly', np.float64(0.9904354099961381)], ['formerly', np.float64(0.97655241415905)], ['airy', np.float64(0.9752334010549083)], ['consequently', np.float64(0.9734308447723443)], ['walker', np.float64(0.9726497393953505)], ['hugh', np.float64(0.9711309308552073)], ['schoolmaster', np.float64(0.9683373517831646)], ['positively', np.float64(0.9672964646472894)], ['thereby', np.float64(0.9671843263191118)], ['lone', np.float64(0.9670576584420993)], ['majestic', np.float64(0.9603236949131866)], ['

In [13]:
seed = 1042
num_words = 50

random.seed(seed)
low_target_word = random.sample(low_shift_words, num_words)
high_target_word = random.sample(high_shift_words, num_words)

print(low_target_word)
print(high_target_word)

random

[['nervous', np.float64(0.5693470723430367)], ['ached', np.float64(0.5824691612551237)], ['chill', np.float64(0.554175758954558)], ['marrying', np.float64(0.5926266744854113)], ['able', np.float64(0.34960250425348444)], ['wheel', np.float64(0.5576721750938181)], ['trees', np.float64(0.40642773677660005)], ['crossed', np.float64(0.562606846642281)], ['country', np.float64(0.5310534508049704)], ['harm', np.float64(0.5482031339962677)], ['bewilderment', np.float64(0.5635864777871481)], ['smiled', np.float64(0.46513614821666655)], ['clinging', np.float64(0.5790510255311011)], ['bright', np.float64(0.532959985662286)], ['collapsed', np.float64(0.5341297576402443)], ['power', np.float64(0.578741033628624)], ['glistened', np.float64(0.5595704570619211)], ['bet', np.float64(0.5040401808317025)], ['friends', np.float64(0.4732909883843714)], ['tone', np.float64(0.5287018153056542)], ['bathroom', np.float64(0.41060132469443894)], ['reign', np.float64(0.5321545969800137)], ['forgotten', np.float64

<module 'random' from '/usr/lib/python3.12/random.py'>

In [14]:
models = []
count = 0
for decade in selected_decades:
  model = [decade]
  vectors = np.load(f"/content/eng-fiction-all/sgns/{decade}-w.npy")
  with open(f"/content/eng-fiction-all/sgns/{decade}-vocab.pkl", "rb") as file:
    keys = p.load(file)

  model.append(keys)
  model.append(KeyedVectors(vector_size=vectors.shape[1]))

  model[2].add_vectors(keys, vectors)
  models.append(model)

##Model Testing

In [268]:
def sanitize_list(output_list, model):
  t10ChatGPT = []
  for word in output_list:
    if word in model[1] and np.mean(model[2][word]) != 0:
      t10ChatGPT.append(word)
  return t10ChatGPT

def cosine_similarity(target_word, generated_words, model):
  target_vector = model[2][target_word]

  generated_vectors = [model[2][word] for word in generated_words]

  cos_sim_list = model[2].cosine_similarities(target_vector, generated_vectors)

  return np.mean(cos_sim_list)

In [2]:
with open("word_generations_high.pkl", "rb") as file:
  word_generations_high = p.load(file)
print(word_generations_high)

FileNotFoundError: [Errno 2] No such file or directory: 'word_generations_high.pkl'

In [1]:
cos_sim_high = {}
for target_word in word_generations_high:
  count_dict = {}
  for count in word_generations_high[target_word]:
    decade_dict = {}
    for decade in selected_decades:
      model = models[selected_decades.index(decade)]
      temp_dict = {}
      allowed_prompts = ["ahistorical", "modern", "historical", str(decade)]
      for prompt in allowed_prompts:
        raw_words = word_generations_high[target_word][count][prompt]
        generated_words = sanitize_list(raw_words, model)
        if len(generated_words) == 0:
          temp_dict[prompt] = np.nan
        else:
          temp_dict[prompt] = cosine_similarity(
              target_word,
              generated_words,
              model
          )
      decade_dict[decade] = temp_dict
    count_dict[count] = decade_dict
cos_sim_high[target_word] = count_dict

print(cos_sim_high)

NameError: name 'word_generations_high' is not defined

In [274]:
rows = []
for target_word in cos_sim_high:
  for count in cos_sim_high[target_word]:
    for decade in cos_sim_high[target_word][count]:
      for prompt in cos_sim_high[target_word][count][decade]:
        avg_score = cos_sim_high[target_word][count][decade][prompt]
        rows.append({
            "target_word": target_word,
            "count": count,
            "decade": decade,
            "prompt": prompt,
            "avg_cos_sim": avg_score
        })

df_high = pd.DataFrame(rows)
df_high

df1 = df_high.pivot_table(
    index="prompt",
    columns="decade",
    values="avg_cos_sim",
    aggfunc="mean"
)

df_high["prompt"] = df_high["prompt"].astype(str)

df_high["prompt"] = df_high["prompt"].replace({
    "1950": "historical_decade",
    "1960": "historical_decade",
    "1970": "historical_decade",
    "1980": "historical_decade",
    "1990": "historical_decade"
})

table_high = df_high.pivot_table(
    index="prompt",
    columns="decade",
    values="avg_cos_sim",
    aggfunc="mean"
)

row_order = ["ahistorical", "modern", "historical", "historical_decade"]
table_high = table_high.loc[row_order]

table_high.round(4)

display(Markdown("#Semantic Neighbors' Average Cosine Similarity Value"))
display(table_high)

#Semantic Neighbors' Average Cosine Similarity Value

decade,1950,1960,1970,1980,1990
prompt,,,,,
ahistorical,0.183719,0.178866,0.192491,0.186725,0.188839
modern,0.177465,0.172726,0.183540,0.182222,0.182803
historical,0.185385,0.180042,0.191543,0.189651,0.189972
historical_decade,0.184677,0.177789,0.189280,0.185053,0.186862


In [273]:
delta_high = table_high.drop(columns=["Mean"]).subtract(
    table_high.loc["ahistorical"].drop("Mean"),
    axis=1
)

delta_high = delta_high.drop(index="ahistorical")

delta_high.round(4)

display(Markdown("#Delta in Semantic Neighbors Average Cosine Similarity Value"))
display(delta_high)

#Delta in Semantic Neighbors Average Cosine Similarity Value

decade,1950,1960,1970,1980,1990
prompt,,,,,
modern,-0.006254,-0.006139,-0.008951,-0.004503,-0.006036
historical,0.001666,0.001177,-0.000948,0.002926,0.001132
historical_decade,0.000957,-0.001076,-0.003211,-0.001672,-0.001977


In [271]:
with open("cos_sim_high.pkl", "wb") as file:
  p.dump(cos_sim_high, file)
print(cos_sim_high)

{'wherever': {0: {1950: {'ahistorical': np.float32(0.18303211), 'modern': np.float32(0.16090406), 'historical': np.float32(0.18852817), '1950': np.float32(0.16824639)}, 1960: {'ahistorical': np.float32(0.1715732), 'modern': np.float32(0.13832623), 'historical': np.float32(0.17031415), '1960': np.float32(0.13832623)}, 1970: {'ahistorical': np.float32(0.17727415), 'modern': np.float32(0.13357982), 'historical': np.float32(0.16405746), '1970': np.float32(0.109700255)}, 1980: {'ahistorical': np.float32(0.12540609), 'modern': np.float32(0.094456814), 'historical': np.float32(0.14771253), '1980': np.float32(0.07018612)}, 1990: {'ahistorical': np.float32(0.110198826), 'modern': np.float32(0.08343455), 'historical': np.float32(0.15210795), '1990': np.float32(0.12343532)}}, 1: {1950: {'ahistorical': np.float32(0.1152833), 'modern': np.float32(0.16459306), 'historical': np.float32(0.18414304), '1950': np.float32(0.18916257)}, 1960: {'ahistorical': np.float32(0.12318877), 'modern': np.float32(0.1